# Explainable Fake News Detection — DL Proof (LSTM & BERT)

This notebook provides Deep Learning cells you can show in viva.
- Baseline we trained: TF-IDF + Logistic Regression (classic ML, fast on CPU).
- DL options below: LSTM (RNN) and BERT (Transformer). Prefer running on Colab GPU.

Tip: For a quick local sanity check, use small subset sizes and 1 epoch.

## 1) Load dataset (ISOT merged)
We reuse `data/raw/news.csv` with columns `text,label` (values `real` or `fake`).

In [ ]:
import os, pandas as pd
data_path = 'data/raw/news.csv'
assert os.path.exists(data_path), f'Missing {data_path}. Ensure the merged CSV exists.'
df = pd.read_csv(data_path)[['text','label']].dropna()
print(df.shape)
print(df['label'].value_counts())

## 2) LSTM (Deep Learning, RNN)
LSTM = Long Short-Term Memory. Minimal PyTorch text classifier for binary labels.
Uncomment the cell below if you want to run locally (needs `torch`).

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import numpy as np

class SimpleDataset(Dataset):
    def __init__(self, texts, labels, vocab=None, max_len=200):
        self.max_len = max_len
        self.labels = labels.str.lower().map({'fake':0,'real':1}).values
        tokens = [t.lower().split() for t in texts]
        if vocab is None:
            cnt = Counter(w for row in tokens for w in row)
            vocab = {w:i+2 for i,(w,_) in enumerate(cnt.most_common(20000))}
            vocab['<pad>']=0; vocab['<unk>']=1
        self.vocab = vocab
        ids = [[vocab.get(w,1) for w in row][:max_len] for row in tokens]
        self.ids = [x + [0]*(max_len-len(x)) for x in ids]
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return torch.tensor(self.ids[i]), torch.tensor(self.labels[i], dtype=torch.long)

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden*2, 2)
    def forward(self, x):
        x = self.emb(x)
        out,_ = self.lstm(x)
        return self.fc(out[:, -1, :])

subset = df.sample(1000, random_state=42)
ds = SimpleDataset(subset['text'], subset['label'])
dl = DataLoader(ds, batch_size=32, shuffle=True)
model = LSTMClassifier(vocab_size=len(ds.vocab))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
model.train()
for epoch in range(1):
    total=0; corr=0
    for x,y in dl:
        opt.zero_grad(); out=model(x); loss=loss_fn(out,y); loss.backward(); opt.step()
        pred=out.argmax(1); total+=len(y); corr+=(pred==y).sum().item()
    print('epoch acc=', corr/total)

## 3) BERT (Deep Learning, Transformer)
BERT = Bidirectional Encoder Representations from Transformers. Minimal HF `transformers` fine-tune.
Uncomment to run (needs `transformers`, `datasets`, `torch`).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np

small = df.sample(2000, random_state=42).copy()
small['label_id'] = small['label'].str.lower().map({'fake':0,'real':1})
ds = Dataset.from_pandas(small[['text','label_id']])
model_name = 'distilbert-base-uncased'
tok = AutoTokenizer.from_pretrained(model_name)
def tok_fn(batch):
    return tok(batch['text'], truncation=True, padding='max_length', max_length=256)
tds = ds.train_test_split(test_size=0.2, seed=42).map(tok_fn, batched=True)
cols = ['input_ids','attention_mask','label_id']
tds = tds.remove_columns([c for c in tds['train'].column_names if c not in cols])
tds = tds.rename_column('label_id','labels')
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
args = TrainingArguments(output_dir='results/bert_sanity', per_device_train_batch_size=8,
                         per_device_eval_batch_size=8, num_train_epochs=1,
                         evaluation_strategy='epoch', logging_steps=50, save_strategy='no')
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {'accuracy': float((preds == p.label_ids).mean())}
trainer = Trainer(model=model, args=args, train_dataset=tds['train'], eval_dataset=tds['test'],
                  tokenizer=tok, compute_metrics=compute_metrics)
trainer.train(); print(trainer.evaluate())

## 4) Notes
- Baseline trained and evaluated locally: TF-IDF + Logistic Regression (fast, reproducible).
- DL cells (LSTM/BERT) provided here for proof and future GPU runs (Colab).
- For live demo, we show ML metrics/plots + LIME; for DL, we show these cells and explain compute trade-offs.